# Interactive map helper functions

> The objective of this modeule is to create some helper function to help us set the correct center and zoom level in `folium` (and other) maps.

In [ ]:
#| default_exp maps._helpers

In [ ]:
#| export
import math

import numpy as np
import xarray as xr

from raincell import open_cml_sample

In [ ]:
#| hide
import folium

In [ ]:
cml = open_cml_sample()

:::{.callout-warning}
These functions were originally created for Plotly, which uses Mapbox internally. This is unlike Leaflet, which Folium uses internally. This is why the estimated zoom may differ slightly from the actual required zoom. Please feel free to report any bugs you find.
:::

When creating the map, we want the initial view to be centred and zoomed properly so that all the links are clearly visible at once. While the geopandas.explore function does this automatically, in some cases we might want to create a Map instance and then add geometries or tiles to it. To provide a good user experience, we need to be able to centre and zoom the map to the correct position. We will start by computing the centre.

In [ ]:
#| hide
float(cml["site_0_lat"].values.min())

3.992722

In [ ]:
#| export
def get_center(cml: xr.DataArray | xr.Dataset) -> tuple[float,float]:
    """ Get the center point of a set of CML in OpenSense standard """
    lats = np.concat([cml["site_0_lat"].values, cml["site_1_lat"].values])
    lons = np.concat([cml["site_0_lon"].values, cml["site_1_lon"].values])
    return float(lats.min() + lats.max()) / 2, float(lons.min() + lons.max()) / 2

In [ ]:
get_center(cml)

(4.0407554999999995, 9.743189999999998)

To ensure that all links are clearly visible on the map, it is important to calculate an appropriate zoom level based on the spatial extent of the data. In particular, we need to determine the zoom required to fit the latitude and longitude ranges of the links within the map view.

We will first compute the zoom level required to represent the latitude. The zoom in meters per pixel can be estimated as follows:

$$
\text{meters\_per\_pixel} = \frac{v_0 \cdot \cos\left(lat_{center}\right)}{2^{\text{zoom\_level}}}
$$

We need a zoom such that:

$$
\text{meters\_per\_pixel} = \frac{|lat_{\max} - lat_{\min}| \cdot \text{meters\_per\_degree}}{\text{height\_pixels}}
$$

Thus, the required zoom level for latitude can be computed as:

$$
\text{zoom\_level} = \log_2 \left( \frac{v_0 \cdot \cos\left(lat_{center}\right) \cdot \text{height\_pixels}}{|lat_{\max} - lat_{\min}| \cdot \text{meters\_per\_degree}} \right)
$$

Where:

- $v_0$ = initial resolution (meters/pixel at zoom level 0). It can be obtained from plotly [website](https://docs.mapbox.com/help/glossary/zoom-level/#zoom-levels-and-geographical-distance)
- $\text{height\_pixels}$ = height of the map layout in pixels
- $lat_{\max}, lat_{\min}$ = maximum and minimum latitude in the data
- $\text{meters\_per\_degree}$ = number of meters per degree of latitude (typically 111320 m/degree)
- $lat_{center}$ = center latitude of the bounding box, i.e. $lat_{center} = \frac{lat_{\max} + lat_{\min}}{2}$


Implementing this in python is straightforward.


In [ ]:
#| exporti
def _get_lat_zoom_start(cml, height=800):
    v0 = 78271.484
    equator_decimal_deg_2_m = 111320
    height += 50 # add some margin
    lats = np.concat([cml["site_0_lat"].values, cml["site_1_lat"].values])
    min_lat, max_lat = float(lats.min()), float(lats.max())
    zoom_level = math.log2(v0 * height * math.cos(math.radians((max_lat + min_lat) / 2)) / (abs(max_lat - min_lat) * equator_decimal_deg_2_m))
    return zoom_level - 0.15 # add some margin

Next, we need to estimate the required zoom level for longitude. Unlike latitude, the number of meters per degree of longitude changes with latitude, and we also need to use the map's width (in pixels) instead of its height. The formula for meters per pixel along the longitude direction is:

$$
\text{meters\_per\_pixel} = \frac{|lon_{\max} - lon_{\min}| \cdot \text{meters\_per\_degree} \cdot \cos\left(lat_{center}\right)}{\text{width\_pixels}}
$$

From this, we can derive the formula for the required zoom level:

$$
\text{zoom\_level} = \log_2 \left( \frac{v_0 \cdot \text{width\_pixels}}{|lon_{\max} - lon_{\min}| \cdot \text{meters\_per\_degree}} \right)
$$

where $\text{width\_pixels}$ is the width of the map layout in pixels.

As before, implementing this calculation in Python is straightforward.

In [ ]:
#| export
def _get_lon_zoom_start(cml, width=600):
    v0 = 78271.484
    equator_decimal_deg_2_m = 111320
    width += 50
    lons = np.concat([cml["site_0_lon"].values, cml["site_1_lon"].values])
    lons_min, lons_max = float(lons.min()), float(lons.max())
    zoom_level = math.log2(v0 * width / (abs(lons_max - lons_min) * equator_decimal_deg_2_m))
    return zoom_level - 0.15 # add some margin

So, the final zoom level for the map should be set to the minimum of the zoom levels calculated for latitude and longitude. This approach ensures that all links are fully visible within the map's bounding box, both vertically and horizontally.

In [ ]:
#| export
def get_zoom_start(cml, height=1200, width=1200):
    return min(_get_lat_zoom_start(cml, height), _get_lon_zoom_start(cml, width))

In [ ]:
get_zoom_start(cml)

12.406704259110562

In [ ]:
#| eval:false
folium.Map(get_center(cml), zoom_start=get_zoom_start(cml))

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()